# CS229 L02 — Supervised Learning Setup & Linear Regression

**Stanford CS229 · Spring 2026 · Instructor: Chris**  
[▶ Lecture Video](https://www.youtube.com/watch?v=cmNIMjPYdgM) · [Official Notes](https://cs229.stanford.edu/notes/cs229-notes1.pdf) · [Course Website](https://cs229.stanford.edu/)

---

**How to use this notebook:**  
> 📌 *Lecture:* — instructor's exact words from the transcript  
> 🎯 **Interview:** — Q&A blocks for interview articulation  

---

## 1. The Supervised Learning Setup

> 📌 *Lecture:* "The basics of supervised learning is going to be a hypothesis — a function from some abstract set X to some abstract set Y. What makes it supervised is this training set. We collect some data which are pairs X and Y, and we try to find a hypothesis that generalizes."

**Formal setup:**

| Symbol | Meaning | Example |
|---|---|---|
| $x^{(i)} \in \mathcal{X}$ | Input / feature vector | Lot size, bedrooms, zip code |
| $y^{(i)} \in \mathcal{Y}$ | Output / label / target | House price |
| $n$ | Number of training examples | 1000 houses |
| $d$ | Number of features / dimensions | 3 features |
| $h: \mathcal{X} \to \mathcal{Y}$ | Hypothesis (predictor) | Linear function |
| $\theta$ | Parameters | Weights and bias |

**Training set:** $\{(x^{(1)}, y^{(1)}), (x^{(2)}, y^{(2)}), ..., (x^{(n)}, y^{(n)})\}$

> 📌 *Lecture:* "What we would like is this x and y to come from some set that's representative of the real world. And then later when we're shown new examples that are not from that original set, this thing is going to label it — it's going to look at a new picture of a cat and say 'yes, that's a cat, not a dog.' That generalization is the heart of machine learning."

**Regression vs Classification:**

> 📌 *Lecture:* "If Y is continuous, we call it a regression problem — real numbers, prices. If Y is discrete, it's a classification problem. Classification problems are probably things you end up solving more in machine learning these days — for example, the way chat GPT works is it has a classifier head at the end that is guessing what's the next word."

| Type | $y$ | Loss used | Example |
|---|---|---|---|
| Regression | $y \in \mathbb{R}$ | Squared loss | House price |
| Binary classification | $y \in \{0, 1\}$ | Logistic loss | Spam/not spam |
| Multiclass | $y \in \{0,...,K-1\}$ | Cross-entropy | ImageNet, next token |

> 🎯 **Interview:** *What is the goal of supervised learning?*  
> Find a hypothesis $h: \mathcal{X} \to \mathcal{Y}$ that generalizes — meaning it was trained on a training set but performs well on unseen examples from the same distribution. The key assumption is that the training set is a representative sample of the real world. If that assumption breaks, the model fails. The three design decisions are: (1) hypothesis class — what functions are possible; (2) loss function — how to measure error; (3) optimization — how to find the best parameters.

## 2. The Linear Hypothesis Class

> 📌 *Lecture:* "Among all of those hypotheses out there, we've now really constrained it. It's entirely defined parametrically — entirely defined by these two little parameters. No matter how many houses we see, we're only going to learn those two parameters. It's a huge reduction in the space of functions. We went from uncountably many functions to this really small class of lines."

**1D case (one feature):**
$$h_\theta(x) = \theta_0 + \theta_1 x$$

**The x₀ = 1 convention:** To unify the bias term, we prepend a 1 to every input vector:
$$x = \begin{bmatrix} 1 \\ x_1 \\ x_2 \\ \vdots \\ x_d \end{bmatrix}, \quad \theta = \begin{bmatrix} \theta_0 \\ \theta_1 \\ \theta_2 \\ \vdots \\ \theta_d \end{bmatrix}$$

Then the hypothesis simplifies to a pure dot product:
$$h_\theta(x) = \theta^T x = \sum_{j=0}^{d} \theta_j x_j$$

> 📌 *Lecture:* "We call them linear — it's a little bit of an abuse of terminology — because of a convention. This is technically an affine function because it has this little offset here. But by the x₀ = 1 convention, we don't have to special-case the θ₀ term and we can easily extend from three to however many dimensions."

**Why linear models matter in 2026:**
- The final layer of every neural network is a linear classifier (logits = Wh)
- Linear models have provable generalization guarantees
- They are the baseline — if a linear model can solve your problem, use it

> 🎯 **Interview:** *Why do we prepend x₀ = 1 to the input vector?*  
> It's a notational convenience that unifies the bias term $\theta_0$ with the weights $\theta_1, ..., \theta_d$ into a single vector $\theta$. This allows us to write $h_\theta(x) = \theta^T x$ as a simple dot product, which is cleaner mathematically and maps directly to vectorized computation. Without this convention, we'd have to handle $\theta_0$ as a special case in every derivation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The x0=1 convention
def add_bias_column(X):
    """Prepend column of 1s — the x0=1 convention."""
    n = X.shape[0]
    return np.hstack([np.ones((n, 1)), X])

# Example: Ames housing data (synthetic)
np.random.seed(42)
n = 100
lot_size = np.random.uniform(3000, 15000, n)
bedrooms = np.random.randint(2, 6, n)
true_theta = np.array([50000, 8.5, 12000])  # [bias, lot_coeff, bedroom_coeff]

X_raw = np.column_stack([lot_size, bedrooms])
X = add_bias_column(X_raw)  # n x 3, with x0=1 prepended
y = X @ true_theta + np.random.randn(n) * 20000  # price with noise

print(f"X shape: {X.shape}  (n={n} examples, d+1={X.shape[1]} features incl. bias)")
print(f"theta shape: {true_theta.shape}")
print(f"\nFirst example:")
print(f"  x = {X[0]}  (x0=1, lot_size={lot_size[0]:.0f}, bedrooms={bedrooms[0]})")
print(f"  y = ${y[0]:,.0f}")
print(f"  h(x) = theta^T x = {X[0] @ true_theta:,.0f}")

## 3. The Least Squares Loss

> 📌 *Lecture:* "Here comes least squares. You have this idea here J(θ) — that's going to be the loss you incur. It's broken down into terms on every single element of the data set. You have $h_\theta(x^{(i)}) - y^{(i)}$ — that's the prediction error. Then we square it, because we want a non-negative number. And we try to pick the θ that minimizes it."

$$J(\theta) = \frac{1}{2} \sum_{i=1}^{n} \left( h_\theta(x^{(i)}) - y^{(i)} \right)^2 = \frac{1}{2} \sum_{i=1}^{n} \left( \theta^T x^{(i)} - y^{(i)} \right)^2$$

The $\frac{1}{2}$ is convention — cancels with the 2 from the derivative:

> 📌 *Lecture:* "The 1/2 is there just by convention. The arg min is insensitive to constants. When we compute the derivative in a minute, these things are going to cancel and that makes things a little bit nicer. It's kind of like the cooking show view of math where we set it up so that it looks nice."

**Why squared loss?**

> 📌 *Lecture:* "Why do we square? Two has a nice property — we can solve exactly. When we compute all the derivatives, the gradient has a really nice form and that lets us solve it exactly. Also historically, Gauss was doing least squares because of that computational advantage — he could predict all kinds of nice things. It was an easy problem."

| Reason | Explanation |
|---|---|
| Non-negative | Error is always ≥ 0 regardless of sign of $(h - y)$ |
| Differentiable everywhere | Unlike absolute value (L1) which has a kink at 0 |
| Closed-form solution | $\nabla J = 0$ can be solved exactly (normal equations) |
| MLE justification | Squared loss = MLE under Gaussian noise assumption (L03) |

**What about other powers?**  
- L1 loss: $\sum |h - y|$ → robust to outliers, but gradient has kink at 0
- L4 loss: $\sum (h - y)^4$ → penalizes large errors more heavily
- Squared is the right choice when errors are Gaussian (which is a reasonable assumption for many regression problems)

> 📌 *Lecture:* "This has a really fancy name: empirical risk minimization. You'll sometimes hear me slip and say ERM, and that's what it means."

**ERM:** $\hat{\theta} = \arg\min_\theta J(\theta) = \arg\min_\theta \frac{1}{2n} \sum_{i=1}^n (h_\theta(x^{(i)}) - y^{(i)})^2$

> 🎯 **Interview:** *Why do we use squared loss for regression instead of absolute value?*  
> Three reasons: (1) differentiable everywhere — absolute value has a non-differentiable kink at zero which complicates optimization; (2) closed-form solution exists — setting $\nabla J = 0$ gives the normal equations $\theta^* = (X^TX)^{-1}X^Ty$, no iterative algorithm needed; (3) probabilistic justification — squared loss is the negative log-likelihood under a Gaussian noise model: if $y = \theta^Tx + \epsilon$ where $\epsilon \sim \mathcal{N}(0, \sigma^2)$, then MLE on the log-likelihood gives exactly the least squares objective. L1 loss corresponds to a Laplace noise model and is more robust to outliers.

## 4. Batch Gradient Descent

> 📌 *Lecture:* "We're going to iterate and walk down in the direction until we get to the bottom. We start with an initial guess — could be a random number, could be zero. We compute the gradient and we walk in the opposite direction. How far are we going to walk? This thing here is called a step size."

**The update rule (batch GD):**
$$\theta := \theta - \alpha \nabla_\theta J(\theta)$$

**Computing the gradient of J:**

For a single example:
$$\frac{\partial}{\partial \theta_j} J(\theta) = (h_\theta(x) - y) \cdot x_j$$

Over the full training set:
$$\frac{\partial}{\partial \theta_j} J(\theta) = \sum_{i=1}^{n} (h_\theta(x^{(i)}) - y^{(i)}) \cdot x_j^{(i)}$$

> 📌 *Lecture:* "This error term — you basically always have this in a ton of these rules. You have the error, what you have, and then you have it multiplied by something here — just the gradient of the function. You'll see this form again and again."

**The key pattern:** gradient = error × feature
$$\nabla_{\theta_j} J = (\hat{y} - y) \cdot x_j$$

This pattern reappears in logistic regression, neural networks, and softmax. It's always: prediction error × local gradient.

**Learning rate tradeoffs:**

> 📌 *Lecture:* "The step size is basically how sure you are in that information. If you take steps that are too small, you just don't get to the optimum fast enough. If you take steps that are too big, you bounce around from both sides. You take a step but shoot past the optimal."

| $\alpha$ too small | $\alpha$ just right | $\alpha$ too large |
|---|---|---|
| Slow convergence | Smooth descent to minimum | Oscillates / diverges |
| Many iterations needed | — | May never converge |

> 🎯 **Interview:** *What is gradient descent and what does the learning rate control?*  
> Gradient descent iteratively updates parameters in the direction of steepest descent: $\theta := \theta - \alpha \nabla J(\theta)$. The learning rate $\alpha$ controls the step size. Too small: converges but slowly. Too large: overshoots the minimum and oscillates or diverges. For convex functions (like least squares), gradient descent with a sufficiently small $\alpha$ is guaranteed to converge to the global minimum. For non-convex functions (like neural networks), it may converge to a local minimum or saddle point — but in practice, modern models trained with SGD generalize well despite this.

In [ ]:
def gradient_descent(X, y, alpha=1e-8, n_iter=1000):
    n, d = X.shape
    theta = np.zeros(d)
    losses = []
    for _ in range(n_iter):
        error = X @ theta - y                       # shape (n,)
        grad = X.T @ error / n                      # shape (d,)  — error × feature, summed
        theta = theta - alpha * grad
        loss = 0.5 * np.mean(error**2)
        losses.append(loss)
    return theta, losses

theta_gd, losses = gradient_descent(X, y, alpha=1e-9, n_iter=2000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(losses)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Loss J(θ)')
ax1.set_title('Batch Gradient Descent — Loss Curve')
ax1.set_yscale('log')

# Compare predictions
y_pred = X @ theta_gd
ax2.scatter(y / 1e3, y_pred / 1e3, alpha=0.4)
ax2.plot([y.min()/1e3, y.max()/1e3], [y.min()/1e3, y.max()/1e3], 'r--')
ax2.set_xlabel('True price ($K)')
ax2.set_ylabel('Predicted price ($K)')
ax2.set_title('Predictions vs Ground Truth')
plt.tight_layout()
plt.show()
print(f"Learned theta: {theta_gd}")
print(f"True theta:    {true_theta}")

## 5. Stochastic Gradient Descent (SGD)

> 📌 *Lecture:* "If I give you the entire internet and ask you to predict the next word in the entire internet, you have to scan the entire internet every time you update your model. That seems really slow. So machine learning people started to try and get algorithms which were even simpler — even dumber. Don't worry, machine learning has got you covered. We have a lot more dumb algorithms."

**Mini-batch SGD update rule:**
$$\theta := \theta - \alpha_B \cdot \frac{1}{B} \sum_{i \in \mathcal{B}} \nabla_\theta \ell(x^{(i)}, y^{(i)}; \theta)$$

where $\mathcal{B}$ is a random mini-batch of size $B \ll n$.

> 📌 *Lecture:* "Stochastic gradient descent — or in the old days, incremental gradient — goes back at least to the 1950s, Robbins and Monro. It's an old classical algorithm. People rediscover it every 10-15 years. It is the workhorse. If you do backprop in PyTorch, you've used mini-batch — the entire system is set up so that you're going to take a small batch of data."

**Why SGD works:**
- Each mini-batch is an **unbiased estimate** of the full gradient
- Noise from sampling actually helps escape local minima (implicit regularization)
- Enables training on datasets too large to fit in memory
- Vectorized batch computation maps to GPU parallelism efficiently

### 5.1 Batch Size — The Deep Question

> 📌 *Lecture:* "Batch size was actually one of the things that broke me intellectually. There was a paper from Facebook labs that showed larger batches get better generalization — their training loss was worse, but they were generalizing better to the real world. As a mathematical person, at first I was like, this is heresy. But that changed the mathematics of it for me."

> 📌 *Lecture:* "The practice is: batch size is how much GPU memory you have. That's basically what people do for systems reasons. And there's a little bit of tuning that people do underneath, but it's a little bit folklore if I'm honest."

| Batch size | Gradient estimate | Compute efficiency | Generalization |
|---|---|---|---|
| $B = 1$ | High variance | Poor (no parallelism) | Often good (noisy = implicit regularizer) |
| $B = 32$–$512$ | Medium variance | Good | Good |
| $B = n$ (full batch) | Zero variance | Limited by RAM | Can overfit more easily |

> 📌 *Lecture:* "The reason I hammer on this so hard is a lot of machine learning, if you look at it, is this trade-off between kind of how we compute things and kind of how we predict them. Machine learning to me really intellectually came into its own as a field when it broke away a little bit from stats and started to train these crazy large models."

**Mini-batch ordering matters:**

> 📌 *Lecture:* "What do you want in that batch? You want it to be a sample of the population. If you showed all the cats first, then all the dogs — it would learn some trivial surface that was only predicting cats. Then it would see the dogs and race to the other side. So mini batches should reflect the overall data set."

**Best practice:** shuffle the dataset each epoch, then take consecutive chunks as batches (without-replacement sampling).

> 🎯 **Interview:** *Why does SGD generalize better than full-batch gradient descent for deep learning?*  
> The noise in SGD from random mini-batches acts as an implicit regularizer — it prevents the model from memorizing the training set. There's also evidence (the Facebook paper on large-batch training) that smaller batches find "flatter" minima that generalize better to the test set, while full-batch GD finds "sharper" minima that are closer to the training optimum but don't transfer as well. The intuition: flat minima correspond to representations that are robust to small input perturbations; sharp minima are fragile. In practice, batch size is often chosen based on GPU memory constraints, not purely statistical grounds.

In [ ]:
def sgd(X, y, alpha=1e-8, batch_size=16, n_epochs=50, seed=42):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    theta = np.zeros(d)
    losses = []
    for epoch in range(n_epochs):
        idx = rng.permutation(n)          # shuffle each epoch
        for start in range(0, n, batch_size):
            batch = idx[start:start + batch_size]
            Xb, yb = X[batch], y[batch]
            error = Xb @ theta - yb
            grad = Xb.T @ error / len(batch)
            theta = theta - alpha * grad
        loss = 0.5 * np.mean((X @ theta - y)**2)
        losses.append(loss)
    return theta, losses

# Compare batch GD vs SGD vs different batch sizes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for bs, label in [(1, 'B=1 (pure SGD)'), (16, 'B=16'), (n, 'Full batch GD')]:
    _, losses_sgd = sgd(X, y, alpha=1e-9, batch_size=bs, n_epochs=100)
    axes[0].plot(losses_sgd, label=label, alpha=0.8)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('SGD: Effect of Batch Size on Loss Curve')
axes[0].legend()
axes[0].set_yscale('log')

# Visualize the noisy gradient path for small batch vs large
# Show loss landscape + path for 1D case
theta_vals = np.linspace(-2e5, 4e5, 200)
X1d = X[:, :2]  # just bias + lot_size
true_t = true_theta[:2]
losses_1d = [0.5 * np.mean((X1d @ np.array([t, true_t[1]]) - y)**2) for t in theta_vals]
axes[1].plot(theta_vals, losses_1d)
axes[1].set_xlabel('θ₀ (bias)')
axes[1].set_ylabel('J(θ)')
axes[1].set_title('Loss landscape — convex bowl for linear regression')
axes[1].axvline(true_t[0], color='r', linestyle='--', label=f'True θ₀={true_t[0]}')
axes[1].legend()

plt.tight_layout()
plt.show()
print("Key insight: for linear regression the loss is a convex bowl — GD always finds the global minimum.")
print("For neural networks: non-convex — multiple local minima, saddle points.")

## 6. The Normal Equations — Closed-Form Solution

> 📌 *Lecture:* "The normal equations are an excuse for me to give you some notation about matrices and vectors. If you look at modern models — they have thousands, hundreds of thousands, millions, billions of parameters. That's really hard to visualize. So we use vector notation to think about it and understand how to manipulate it."

**Matrix form:** Stack all examples into a design matrix:

$$X = \begin{bmatrix} - (x^{(1)})^T - \\ - (x^{(2)})^T - \\ \vdots \\ - (x^{(n)})^T - \end{bmatrix} \in \mathbb{R}^{n \times (d+1)}, \quad y = \begin{bmatrix} y^{(1)} \\ y^{(2)} \\ \vdots \\ y^{(n)} \end{bmatrix} \in \mathbb{R}^n$$

The loss function in matrix form:
$$J(\theta) = \frac{1}{2} \|X\theta - y\|^2 = \frac{1}{2}(X\theta - y)^T(X\theta - y)$$

**Derivation — set gradient to zero:**

$$\nabla_\theta J = X^T(X\theta - y) = X^TX\theta - X^Ty = 0$$

$$\boxed{\theta^* = (X^TX)^{-1}X^Ty}$$

This is the **normal equation** — the closed-form solution for linear regression.

> 📌 *Lecture:* "I cheated in one part of this derivation. I assumed something about X when I did this." → *Student: "Assumed that XᵀX is invertible."* → "Exactly right. If I had relatively few examples in a huge set of dimensions, this would be nonsensical."

**When does $(X^TX)^{-1}$ not exist?**
- $n < d+1$: fewer examples than parameters (underdetermined system)
- Multicollinearity: two features are perfectly correlated (linear dependence)
- In these cases: $X^TX$ has a null space, and there are infinitely many $\theta^*$ all giving the same loss

**Fix:** Ridge regression adds $\lambda I$: $\theta^* = (X^TX + \lambda I)^{-1}X^Ty$ — always invertible when $\lambda > 0$

> 🎯 **Interview:** *When would you use normal equations vs gradient descent?*  
> Normal equations give the exact solution in one step — no hyperparameters, no learning rate to tune. But they require computing $(X^TX)^{-1}$, which is $O(d^3)$ for a $d$-dimensional problem. For $d > 10{,}000$ this becomes prohibitively expensive. Gradient descent scales to millions of parameters because each update is just a matrix-vector multiply $O(nd)$. So: normal equations for small $d$ (tabular ML, small regression problems); gradient descent for anything involving neural networks or large feature spaces.

In [ ]:
def normal_equations(X, y):
    """Closed-form: theta* = (X^T X)^{-1} X^T y"""
    return np.linalg.solve(X.T @ X, X.T @ y)  # more numerically stable than explicit inverse

theta_ne = normal_equations(X, y)

# Compare all three methods
theta_sgd_final, _ = sgd(X, y, alpha=1e-9, batch_size=16, n_epochs=500)

print("Parameter estimates comparison:")
print(f"True theta:        {true_theta}")
print(f"Normal equations:  {theta_ne.round(1)}")
print(f"SGD (500 epochs):  {theta_sgd_final.round(1)}")
print()

# Residuals
for name, theta in [("Normal equations", theta_ne), ("SGD", theta_sgd_final)]:
    y_pred = X @ theta
    rmse = np.sqrt(np.mean((y_pred - y)**2))
    print(f"{name} RMSE: ${rmse:,.0f}")

# Check XTX invertibility
XTX = X.T @ X
print(f"\nX^T X rank: {np.linalg.matrix_rank(XTX)} (d+1={X.shape[1]})")
print(f"X^T X is invertible: {np.linalg.matrix_rank(XTX) == X.shape[1]}")
print(f"Condition number: {np.linalg.cond(XTX):.2e}  (large = near-singular)")

## 7. The Big Picture — What L02 Reveals

> 📌 *Lecture:* "What boils down to is basically solving a set of these equations — that arg min. Being able to take a loss function and run an optimization procedure — fancy one called backprop that you'll learn in the course — underpins pretty much 99% of the models you're likely to encounter today."

> 📌 *Lecture:* "One of the things that made machine learning so powerful is this trade-off between kind of how we compute things and kind of how we predict them. When machine learning really intellectually came into its own was when it broke away a little bit from stats and started to train these crazy large models — and understood what were the computational limits of what kind of models we could have."

**The thread from linear regression to LLMs:**

| Concept | Linear Regression | Modern LLMs |
|---|---|---|
| Hypothesis class | $h_\theta(x) = \theta^Tx$ | Transformer with billions of parameters |
| Loss function | $\frac{1}{2}(\hat{y} - y)^2$ | Cross-entropy $-\log p(y|x)$ |
| Optimization | SGD | Adam/AdamW + gradient clipping |
| Data | 1000 house prices | Trillions of tokens |
| Parameters | 3 scalars | 70B+ floats |

> 📌 *Lecture:* "In statistics, we're very interested in when the optimization problem recovers the right answer. In AI we're like 'we ran out of compute credits, we'll stop — feels good, let's go to the next model.' And that's crazy enough how it works."

> 🎯 **Interview:** *What is empirical risk minimization and why is it the foundation of supervised learning?*  
> ERM says: find the hypothesis that minimizes the average loss on the training set — $\hat{\theta} = \arg\min_\theta \frac{1}{n} \sum_i \ell(h_\theta(x^{(i)}), y^{(i)})$. The key assumption is that the training set is drawn i.i.d. from the same distribution as the test data. If this holds, minimizing training loss generalizes to test performance. ERM underpins everything from linear regression to neural networks — the only things that change are the hypothesis class and the loss function. The optimization algorithm (GD, SGD, Adam) is just the mechanism for finding the ERM solution.

---

## External Resources

| Resource | What it covers | When to use |
|---|---|---|
| [CS229 Notes 1 — Supervised Learning](https://cs229.stanford.edu/notes/cs229-notes1.pdf) | Full derivation with MLE motivation | Authoritative reference for this lecture |
| [3Blue1Brown — Gradient Descent](https://www.youtube.com/watch?v=IHZwWFHWa-w) | Best visual intuition for GD | After this notebook |
| [Andrej Karpathy — micrograd](https://github.com/karpathy/micrograd) | Build gradient descent from scratch | Implementation depth |
| [The Matrix Cookbook](https://www.math.uwaterloo.ca/~hwolkowi/matrixcookbook.pdf) | Matrix derivative identities — how $\nabla_\theta \|X\theta - y\|^2 = 2X^T(X\theta - y)$ | Reference during derivations |
| [An Introduction to Statistical Learning (ISLR) Ch.3](https://www.statlearning.com/) | Statistical treatment of linear regression | Bias-variance in regression |
| [Revisiting Small Batch Training (Facebook 2018)](https://arxiv.org/abs/1804.07612) | The batch size generalization paper Chris mentioned | Deep dive on batch size |
| [Robbins & Monro 1951](https://www.jstor.org/stable/2236626) | Original SGD paper ("stochastic approximation") | Historical origin of SGD |